# Prepare BMMC RNA–Protein ratio splits

This notebook adapts the previous RNA–ATAC split pipeline to RNA–Protein / CITE-seq data.

Main changes:
- split modalities by `feature_types == 'GEX'` and `feature_types == 'ADT'`;
- preprocess RNA with standard QC + normalization + HVG selection;
- preprocess Protein/ADT with a `counts` layer + CLR normalization using `muon.prot.pp.clr`;
- optionally map CD/ADT marker names to HGNC-approved gene symbols;
- generate the same train/validation and partial-pairing ratio folders.


In [12]:
# =========================
# 0. Imports and parameters
# =========================
from __future__ import annotations

import json
import os
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.model_selection import train_test_split

try:
    from muon import prot as pt
except Exception as e:
    pt = None
    print("Warning: failed to import muon.prot. Protein CLR preprocessing will fail unless muon is installed.")
    print(repr(e))


# -------------------------
# Paths
# -------------------------
INPUT_H5AD = "/data5/zhangye/scMRDR/input/BMMC/raw_input/GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad"
OUTPUT_DIR = "/data5/zhangye/scMRDR/input/BMMC/preprocessed_input/RNA_PROTEIN"

# Optional HGNC / CD marker mapping file.
# Set to None if you do not want to map protein marker names.
HGNC_CD_MAPPING_CSV = "/data5/zhangye/scMRDR/scripts/BMMC/CD_gene/group-471.csv"


# -------------------------
# Split parameters
# -------------------------
SEED = 1234
VAL_FRAC = 0.2
SINGLE_FRACS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
RNA_KEEP_PROB = 0.5  # among single-modality cells, probability of keeping RNA rather than Protein


# -------------------------
# RNA preprocessing parameters
# -------------------------
BATCH_KEY = "batch"
CELL_TYPE_KEY = "cell_type"
MT_PREFIX = "MT-"

RNA_MIN_GENES = 200
RNA_MAX_GENES = 5000
RNA_MAX_COUNTS = 15000
RNA_MAX_MT = 20
RNA_MIN_CELLS_PER_GENE = 3

RNA_HVG_MIN_MEAN = 0.02
RNA_HVG_MAX_MEAN = 4
RNA_HVG_MIN_DISP = 0.5


# -------------------------
# Protein preprocessing parameters
# -------------------------
PROTEIN_MIN_CELLS = 1       # keep ADTs detected in at least this many cells
PROTEIN_MIN_COUNTS_CELL = 0 # set >0 if you want to drop cells with no protein counts


# -------------------------
# Split sanity checks
# -------------------------
MIN_TRAIN_RNA_CELLS = 50
MIN_TRAIN_PROTEIN_CELLS = 50
MIN_VAL_QUERY_CELLS = 20


In [13]:
# =========================
# 1. Optional cell-type mapping
# =========================
# This follows your coarse BMMC label grouping.
# If CELL_TYPE_KEY is unavailable, the code will simply skip this mapping.
final_celltype_mapping = {
    # CD4 T
    'CD4+ T naive': 'CD4 T',
    'CD4+ T activated': 'CD4 T',
    'CD4+ T activated integrinB7+': 'CD4 T',
    'CD4+ T CD314+ CD45RA+': 'CD4 T',
    'T reg': 'CD4 T',

    # CD8 T
    'CD8+ T naive': 'CD8 T',
    'CD8+ T naive CD127+ CD26- CD101-': 'CD8 T',
    'CD8+ T CD49f+': 'CD8 T',
    'CD8+ T TIGIT+ CD45RO+': 'CD8 T',
    'CD8+ T CD57+ CD45RA+': 'CD8 T',
    'CD8+ T CD69+ CD45RO+': 'CD8 T',
    'CD8+ T TIGIT+ CD45RA+': 'CD8 T',
    'CD8+ T CD69+ CD45RA+': 'CD8 T',
    'CD8+ T CD57+ CD45RO+': 'CD8 T',
    'CD8+ T': 'CD8 T',
    'MAIT': 'CD8 T',
    'gdT TCRVD2+': 'CD8 T',
    'gdT CD158b+': 'CD8 T',
    'dnT': 'CD8 T',

    # B cells
    'Naive CD20+ B IGKC+': 'B cell',
    'Naive CD20+ B IGKC-': 'B cell',
    'Naive CD20+ B': 'B cell',
    'B1 B IGKC+': 'B cell',
    'B1 B IGKC-': 'B cell',
    'B1 B': 'B cell',
    'Transitional B': 'B cell',

    # Plasma cells
    'Plasmablast IGKC+': 'Plasma cell',
    'Plasmablast IGKC-': 'Plasma cell',
    'Plasma cell IGKC+': 'Plasma cell',
    'Plasma cell IGKC-': 'Plasma cell',
    'Plasma cell': 'Plasma cell',

    # NK
    'NK': 'NK',
    'NK CD158e1+': 'NK',

    # Mono
    'CD14+ Mono': 'Mono',
    'CD16+ Mono': 'Mono',

    # DC
    'pDC': 'DC',
    'cDC1': 'DC',
    'cDC2': 'DC',

    # ILC
    'ILC': 'ILC',
    'ILC1': 'ILC',

    # Progenitors
    'HSC': 'Progenitor',
    'Lymph prog': 'Progenitor',
    'G/M prog': 'Progenitor',
    'MK/E prog': 'Progenitor',
    'ID2-hi myeloid prog': 'Progenitor',
    'T prog cycling': 'Progenitor',

    # Erythroid
    'Erythroblast': 'Erythroid',
    'Normoblast': 'Erythroid',
    'Proerythroblast': 'Erythroid',
    'Reticulocyte': 'Erythroid',
}


In [14]:
# =========================
# 2. Utilities
# =========================
def set_seed(seed: int = 1234) -> None:
    random.seed(seed)
    np.random.seed(seed)


def ensure_dir(path: os.PathLike) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)


def safe_write_h5ad(adata: ad.AnnData, path: os.PathLike) -> None:
    """Write AnnData safely by converting np.matrix layers to ndarray."""
    adata = adata.copy()
    if isinstance(adata.X, np.matrix):
        adata.X = np.asarray(adata.X)
    for k in list(adata.layers.keys()):
        if isinstance(adata.layers[k], np.matrix):
            adata.layers[k] = np.asarray(adata.layers[k])
    adata.write(str(path))


def to_dense(x):
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)


def require_obs_column(adata: ad.AnnData, key: str, fill_value: str = "batch0") -> None:
    if key not in adata.obs.columns:
        adata.obs[key] = fill_value


def add_counts_layer_if_missing(adata: ad.AnnData) -> None:
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()


def get_common_names(*arrays: Sequence[str]) -> List[str]:
    if len(arrays) == 0:
        return []
    common = set(arrays[0])
    for arr in arrays[1:]:
        common &= set(arr)
    return sorted(common)


def subset_and_copy(adata: ad.AnnData, cells: Sequence[str], features: Optional[Sequence[str]] = None) -> ad.AnnData:
    out = adata[list(cells)].copy()
    if features is not None:
        out = out[:, list(features)].copy()
    return out


def assign_partial_modality(
    cells: Sequence[str],
    single_frac: float = 0.2,
    rna_keep_prob: float = 0.5,
    seed: Optional[int] = None,
) -> Dict[str, object]:
    """
    Create paired / RNA-only / Protein-only cell assignments.

    single_frac controls the fraction of cells made single-modality.
    Among those single-modality cells, rna_keep_prob controls whether RNA is kept.
    """
    rng = random.Random(seed) if seed is not None else random
    cells = list(cells)
    n = len(cells)
    n_single = round(n * single_frac)

    single_cells = rng.sample(cells, n_single) if n_single > 0 else []
    paired_cells = sorted(set(cells) - set(single_cells))

    modality = {c: "paired" for c in cells}
    for c in single_cells:
        modality[c] = "RNA" if rng.random() < rna_keep_prob else "Protein"

    rna_only = sorted([c for c, m in modality.items() if m == "RNA"])
    protein_only = sorted([c for c, m in modality.items() if m == "Protein"])

    return {
        "modality": modality,
        "paired_cells": paired_cells,
        "rna_only_cells": rna_only,
        "protein_only_cells": protein_only,
    }


def add_coarse_celltype(adata: ad.AnnData, source_key: str = CELL_TYPE_KEY, target_key: str = "celltype") -> None:
    if source_key in adata.obs.columns:
        adata.obs[target_key] = adata.obs[source_key].map(final_celltype_mapping)
        # keep original labels if not present in the mapping
        adata.obs[target_key] = adata.obs[target_key].fillna(adata.obs[source_key].astype(str))


In [15]:
# =========================
# 3. Read RNA and Protein from CITE-seq h5ad
# =========================
def read_bmmc_cite(input_h5ad: os.PathLike) -> Tuple[ad.AnnData, ad.AnnData]:
    """
    Read BMMC CITE-seq AnnData and split it into RNA/GEX and Protein/ADT.

    Preferred split: adata.var['feature_types'] == 'GEX' / 'ADT'.
    """
    adata = sc.read_h5ad(str(input_h5ad))

    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()

    if "cellid" not in adata.obs.columns:
        adata.obs["cellid"] = np.arange(adata.n_obs)

    if "feature_types" not in adata.var.columns:
        raise ValueError("feature_types not found in adata.var. Please add feature_types or manually split RNA/Protein.")

    feature_types = adata.var["feature_types"].astype(str)
    gex_mask = feature_types.eq("GEX")
    adt_mask = feature_types.eq("ADT")

    if gex_mask.sum() == 0 or adt_mask.sum() == 0:
        raise ValueError("feature_types exists, but GEX or ADT features were not found.")

    rna = adata[:, gex_mask.values].copy()
    prot = adata[:, adt_mask.values].copy()

    # Keep modality-relevant embeddings only.
    rna.obsm = {k: rna.obsm[k] for k in ["GEX_X_pca", "GEX_X_umap"] if k in rna.obsm}
    prot.obsm = {
        k: prot.obsm[k]
        for k in ["ADT_X_pca", "ADT_X_umap", "ADT_isotype_controls"]
        if k in prot.obsm
    }

    rna.var_names_make_unique()
    prot.var_names_make_unique()
    return rna, prot


In [16]:
# =========================
# 4. Preprocess RNA
# =========================
def preprocess_rna(rna: ad.AnnData) -> ad.AnnData:
    rna = rna.copy()
    require_obs_column(rna, BATCH_KEY, "batch0")

    rna.var["mt"] = rna.var_names.astype(str).str.startswith(MT_PREFIX)
    sc.pp.calculate_qc_metrics(rna, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

    sc.pp.filter_genes(rna, min_cells=RNA_MIN_CELLS_PER_GENE)
    sc.pp.filter_cells(rna, min_genes=RNA_MIN_GENES)
    rna = rna[rna.obs["n_genes_by_counts"] <= RNA_MAX_GENES, :].copy()
    rna = rna[rna.obs["total_counts"] <= RNA_MAX_COUNTS, :].copy()
    rna = rna[rna.obs["pct_counts_mt"] < RNA_MAX_MT, :].copy()

    rna.layers["counts"] = rna.X.copy()
    sc.pp.normalize_total(rna)
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(
        rna,
        batch_key=BATCH_KEY,
        min_mean=RNA_HVG_MIN_MEAN,
        max_mean=RNA_HVG_MAX_MEAN,
        min_disp=RNA_HVG_MIN_DISP,
    )

    add_coarse_celltype(rna)
    return rna


In [17]:
# =========================
# 5. Optional protein marker name mapping
# =========================
def load_hgnc_cd_mapper(mapping_csv: Optional[os.PathLike]):
    """Return a function that maps ADT marker names to HGNC-approved symbols."""
    if mapping_csv is None:
        return None

    mapping_csv = Path(mapping_csv)
    if not mapping_csv.exists():
        print(f"HGNC mapping file not found, skip protein marker mapping: {mapping_csv}")
        return None

    df_hgnc = pd.read_csv(mapping_csv)
    required = {"Approved symbol", "Previous symbols", "Aliases"}
    missing = required - set(df_hgnc.columns)
    if missing:
        print(f"HGNC mapping file misses columns {missing}, skip protein marker mapping.")
        return None

    df_hgnc[["Previous symbols", "Aliases"]] = df_hgnc[["Previous symbols", "Aliases"]].fillna("")

    lookup = {}
    for _, row in df_hgnc.iterrows():
        approved = str(row["Approved symbol"]).strip()
        if not approved or approved == "nan":
            continue
        candidates = [approved]
        candidates += [s.strip() for s in str(row["Previous symbols"]).split(",") if s.strip()]
        candidates += [s.strip() for s in str(row["Aliases"]).split(",") if s.strip()]
        for c in candidates:
            lookup.setdefault(c, approved)

    def mapper(marker_name: str) -> str:
        return lookup.get(str(marker_name), str(marker_name))

    return mapper


def map_protein_var_names(prot: ad.AnnData, mapping_csv: Optional[os.PathLike]) -> ad.AnnData:
    prot = prot.copy()
    mapper = load_hgnc_cd_mapper(mapping_csv)

    prot.var["cd_name"] = prot.var_names.astype(str)
    if mapper is None:
        prot.var["gene_name"] = prot.var["cd_name"].astype(str)
    else:
        prot.var["gene_name"] = [mapper(x) for x in prot.var["cd_name"].astype(str)]

    prot.var_names = prot.var["gene_name"].astype(str).tolist()
    prot.var_names_make_unique()
    return prot


In [18]:
# =========================
# 6. Preprocess Protein / ADT
# =========================
def preprocess_protein(
    prot: ad.AnnData,
    mapping_csv: Optional[os.PathLike] = HGNC_CD_MAPPING_CSV,
) -> ad.AnnData:
    """
    Protein preprocessing for CITE-seq ADT features.

    Important difference from ATAC:
    - no peak-level filtering;
    - no gene activity construction;
    - keep raw counts in layers['counts'];
    - CLR-normalize ADT values with muon.prot.pp.clr.
    """
    if pt is None:
        raise ImportError("muon.prot is not available. Please install muon before running protein CLR preprocessing.")

    prot = prot.copy()
    require_obs_column(prot, BATCH_KEY, "batch0")

    # Basic protein-level filtering. Keep this mild because ADT feature count is usually small.
    sc.pp.calculate_qc_metrics(prot, percent_top=None, log1p=False, inplace=True)
    if PROTEIN_MIN_CELLS is not None and PROTEIN_MIN_CELLS > 0:
        sc.pp.filter_genes(prot, min_cells=PROTEIN_MIN_CELLS)
    if PROTEIN_MIN_COUNTS_CELL is not None and PROTEIN_MIN_COUNTS_CELL > 0:
        prot = prot[prot.obs["total_counts"] >= PROTEIN_MIN_COUNTS_CELL, :].copy()

    prot.layers["counts"] = prot.X.copy()

    # CLR normalization. This updates prot.X.
    pt.pp.clr(prot)

    # Mark all protein features as highly_variable for downstream code compatibility.
    prot.var["highly_variable"] = True

    # Optional CD/ADT marker-name mapping after CLR.
    prot = map_protein_var_names(prot, mapping_csv)

    add_coarse_celltype(prot)
    return prot


In [19]:
# =========================
# 7. Optional combined feature file
# =========================
def build_feature_aligned_rna_protein(rna_qc: ad.AnnData, prot_qc: ad.AnnData) -> ad.AnnData:
    """
    Build a convenience combined AnnData with outer-joined RNA genes and protein markers.

    Unlike RNA-ATAC gene activity, RNA and Protein do not naturally share one feature space.
    This file is mainly for bookkeeping / visualization. Training split files below are the main outputs.
    """
    rna = rna_qc.copy()
    prot = prot_qc.copy()

    rna.var["modality_feature_type"] = "RNA"
    prot.var["modality_feature_type"] = "Protein"

    combined = ad.concat(
        [rna, prot],
        join="outer",
        label="modality",
        keys=["rna", "protein"],
        index_unique="__",
    )
    hv = rna.var["highly_variable"] if "highly_variable" in rna.var.columns else np.ones(rna.n_vars, dtype=bool)
    combined.uns["rna_hvg"] = list(rna.var_names[hv].astype(str))
    combined.uns["protein_features"] = list(prot.var_names.astype(str))
    return combined


In [20]:
# =========================
# 8. Save split h5ad files
# =========================
def save_split_h5ads(
    outdir: os.PathLike,
    train_rna_ref: ad.AnnData,
    train_protein_ref: ad.AnnData,
    train_protein_full: ad.AnnData,
    val_query_protein: ad.AnnData,
    val_true_rna: ad.AnnData,
    val_true_protein: ad.AnnData,
) -> None:
    outdir = Path(outdir)
    safe_write_h5ad(train_rna_ref, outdir / "train_rna_ref.h5ad")
    safe_write_h5ad(train_protein_ref, outdir / "train_protein_ref.h5ad")
    safe_write_h5ad(train_protein_full, outdir / "train_protein_full.h5ad")
    safe_write_h5ad(val_query_protein, outdir / "val_query_protein.h5ad")
    safe_write_h5ad(val_true_rna, outdir / "val_true_rna.h5ad")
    safe_write_h5ad(val_true_protein, outdir / "val_true_protein.h5ad")


In [21]:
# =========================
# 9. Ratio split generation
# =========================
def generate_splits(
    rna_qc_path: os.PathLike,
    protein_qc_path: os.PathLike,
    out_root: os.PathLike,
    seed: int = 1234,
    val_frac: float = 0.2,
    single_fracs: Sequence[float] = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    rna_keep_prob: float = 0.5,
    min_train_rna_cells: int = 50,
    min_train_protein_cells: int = 50,
    min_val_query_cells: int = 20,
) -> pd.DataFrame:
    rna = sc.read_h5ad(str(rna_qc_path))
    prot = sc.read_h5ad(str(protein_qc_path))

    # RNA QC and Protein QC may remove different cells, so use their intersection.
    common_cells = get_common_names(rna.obs_names.tolist(), prot.obs_names.tolist())
    if len(common_cells) == 0:
        raise ValueError("No common cells shared by RNA_counts_qc and protein_counts_qc.")

    rna = rna[common_cells].copy()
    prot = prot[common_cells].copy()

    all_cells = np.array(common_cells, dtype=object)
    train_cells, val_cells = train_test_split(
        all_cells,
        test_size=val_frac,
        random_state=seed,
        shuffle=True,
    )
    train_cells = np.array(sorted(train_cells.tolist()), dtype=object)
    val_cells = np.array(sorted(val_cells.tolist()), dtype=object)

    ensure_dir(out_root)
    summaries = []

    for sf in single_fracs:
        ratio_label = f"single_{int(round(sf * 100)):03d}"
        outdir = Path(out_root) / ratio_label
        ensure_dir(outdir)

        train_assign = assign_partial_modality(
            train_cells,
            single_frac=sf,
            rna_keep_prob=rna_keep_prob,
            seed=seed + int(round(sf * 1000)) + 11,
        )
        val_assign = assign_partial_modality(
            val_cells,
            single_frac=sf,
            rna_keep_prob=rna_keep_prob,
            seed=seed + int(round(sf * 1000)) + 97,
        )

        # Keep the same logic as RNA-ATAC:
        # RNA reference contains paired + RNA-only cells.
        # Protein reference/query contains paired + Protein-only cells.
        train_rna_ref_cells = sorted(train_assign["paired_cells"] + train_assign["rna_only_cells"])
        train_protein_ref_cells = sorted(train_assign["paired_cells"] + train_assign["protein_only_cells"])
        val_query_protein_cells = sorted(val_assign["paired_cells"] + val_assign["protein_only_cells"])

        if (
            len(train_rna_ref_cells) < min_train_rna_cells
            or len(train_protein_ref_cells) < min_train_protein_cells
            or len(val_query_protein_cells) < min_val_query_cells
        ):
            meta = {
                "ratio_label": ratio_label,
                "single_frac": sf,
                "status": "skipped",
                "reason": "too_few_cells",
                "train_rna_ref": len(train_rna_ref_cells),
                "train_protein_ref": len(train_protein_ref_cells),
                "val_query_protein": len(val_query_protein_cells),
            }
            with open(outdir / "split_info.json", "w", encoding="utf-8") as f:
                json.dump(meta, f, indent=2, ensure_ascii=False)
            summaries.append(meta)
            continue

        # Main outputs.
        train_rna_ref = subset_and_copy(rna, train_rna_ref_cells)
        train_protein_ref = subset_and_copy(prot, train_protein_ref_cells)

        # Full train protein keeps all train cells, useful for methods that expect full query/reference objects.
        train_protein_full = subset_and_copy(prot, train_cells)

        # Query Protein and paired true RNA for protein->RNA prediction/evaluation.
        val_query_protein = subset_and_copy(prot, val_query_protein_cells)
        val_true_rna = subset_and_copy(rna, val_query_protein_cells)
        val_true_protein = subset_and_copy(prot, val_query_protein_cells)

        save_split_h5ads(
            outdir,
            train_rna_ref=train_rna_ref,
            train_protein_ref=train_protein_ref,
            train_protein_full=train_protein_full,
            val_query_protein=val_query_protein,
            val_true_rna=val_true_rna,
            val_true_protein=val_true_protein,
        )

        meta = {
            "ratio_label": ratio_label,
            "single_frac": sf,
            "status": "ok",
            "train_cells": train_cells.tolist(),
            "val_cells": val_cells.tolist(),
            "train_paired_cells": train_assign["paired_cells"],
            "train_rna_only_cells": train_assign["rna_only_cells"],
            "train_protein_only_cells": train_assign["protein_only_cells"],
            "val_paired_cells": val_assign["paired_cells"],
            "val_rna_only_cells": val_assign["rna_only_cells"],
            "val_protein_only_cells": val_assign["protein_only_cells"],
            "train_rna_ref_cells": train_rna_ref_cells,
            "train_protein_ref_cells": train_protein_ref_cells,
            "val_query_protein_cells": val_query_protein_cells,
            "rna_features": rna.var_names.astype(str).tolist(),
            "protein_features": prot.var_names.astype(str).tolist(),
            "n_train_rna_ref_cells": len(train_rna_ref_cells),
            "n_train_protein_ref_cells": len(train_protein_ref_cells),
            "n_val_query_protein_cells": len(val_query_protein_cells),
            "n_rna_features": rna.n_vars,
            "n_protein_features": prot.n_vars,
        }
        with open(outdir / "split_info.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2, ensure_ascii=False)

        summaries.append({
            "ratio_label": ratio_label,
            "single_frac": sf,
            "status": "ok",
            "train_paired": len(train_assign["paired_cells"]),
            "train_rna_only": len(train_assign["rna_only_cells"]),
            "train_protein_only": len(train_assign["protein_only_cells"]),
            "val_paired": len(val_assign["paired_cells"]),
            "val_rna_only": len(val_assign["rna_only_cells"]),
            "val_protein_only": len(val_assign["protein_only_cells"]),
            "train_rna_ref": len(train_rna_ref_cells),
            "train_protein_ref": len(train_protein_ref_cells),
            "val_query_protein": len(val_query_protein_cells),
            "n_rna_features": rna.n_vars,
            "n_protein_features": prot.n_vars,
        })

    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(Path(out_root) / "summary_all_ratios.csv", index=False)
    return summary_df


In [22]:
# =========================
# 10. Run full pipeline
# =========================
def main():
    set_seed(SEED)
    ensure_dir(OUTPUT_DIR)

    outdir = Path(OUTPUT_DIR)
    rna_qc_path = outdir / "RNA_counts_qc.h5ad"
    protein_qc_path = outdir / "protein_counts_qc.h5ad"
    feature_aligned_path = outdir / "feature_aligned_rna_protein.h5ad"
    split_root = outdir / "results_ratio_loop_rna_protein"

    print("Reading BMMC CITE-seq RNA/Protein data...")
    rna_raw, prot_raw = read_bmmc_cite(INPUT_H5AD)
    print("RNA raw:", rna_raw.shape)
    print("Protein raw:", prot_raw.shape)

    print("[1/4] Preprocess RNA")
    rna_qc = preprocess_rna(rna_raw)
    print("RNA QC:", rna_qc.shape)
    safe_write_h5ad(rna_qc, rna_qc_path)

    print("[2/4] Preprocess Protein")
    prot_qc = preprocess_protein(prot_raw, mapping_csv=HGNC_CD_MAPPING_CSV)
    print("Protein QC:", prot_qc.shape)
    safe_write_h5ad(prot_qc, protein_qc_path)

    print("[3/4] Build optional combined RNA-Protein file")
    feature_aligned = build_feature_aligned_rna_protein(rna_qc, prot_qc)
    safe_write_h5ad(feature_aligned, feature_aligned_path)

    print("[4/4] Generate ratio splits")
    summary_df = generate_splits(
        rna_qc_path=rna_qc_path,
        protein_qc_path=protein_qc_path,
        out_root=split_root,
        seed=SEED,
        val_frac=VAL_FRAC,
        single_fracs=SINGLE_FRACS,
        rna_keep_prob=RNA_KEEP_PROB,
        min_train_rna_cells=MIN_TRAIN_RNA_CELLS,
        min_train_protein_cells=MIN_TRAIN_PROTEIN_CELLS,
        min_val_query_cells=MIN_VAL_QUERY_CELLS,
    )
    print(summary_df)
    return summary_df


summary_df = main()


Reading BMMC CITE-seq RNA/Protein data...


/home/zhangye/anaconda3/envs/scmrdr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


RNA raw: (90261, 13953)
Protein raw: (90261, 134)
[1/4] Preprocess RNA
RNA QC: (83290, 13953)
[2/4] Preprocess Protein


/home/zhangye/anaconda3/envs/scmrdr/lib/python3.9/site-packages/muon/_prot/preproc.py:219: UserWarning: adata.X is sparse but not in CSC format. Converting to CSC.
  warn("adata.X is sparse but not in CSC format. Converting to CSC.")


Protein QC: (90261, 134)
[3/4] Build optional combined RNA-Protein file
[4/4] Generate ratio splits
  ratio_label  single_frac status  train_paired  train_rna_only  \
0  single_000          0.0     ok         66632               0   
1  single_020          0.2     ok         53306            6673   
2  single_040          0.4     ok         39979           13350   
3  single_060          0.6     ok         26653           19921   
4  single_080          0.8     ok         13326           26774   
5  single_100          1.0     ok             0           33395   

   train_protein_only  val_paired  val_rna_only  val_protein_only  \
0                   0       16658             0                 0   
1                6653       13326          1708              1624   
2               13303        9995          3440              3223   
3               20058        6663          5071              4924   
4               26532        3332          6673              6653   
5               

## Expected output structure

```text
RNA_PROTEIN/
├── RNA_counts_qc.h5ad
├── protein_counts_qc.h5ad
├── feature_aligned_rna_protein.h5ad
└── results_ratio_loop_rna_protein/
    ├── summary_all_ratios.csv
    ├── single_000/
    │   ├── train_rna_ref.h5ad
    │   ├── train_protein_ref.h5ad
    │   ├── train_protein_full.h5ad
    │   ├── val_query_protein.h5ad
    │   ├── val_true_rna.h5ad
    │   ├── val_true_protein.h5ad
    │   └── split_info.json
    ├── single_020/
    └── ...
```
